<a href="https://colab.research.google.com/github/swalehaparvin/AI-Safety-and-Red-Teaming/blob/main/Tokenization_Layer_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## The Tokenization Layer: How Models Read Text: Tokens vs. Characters

In [ ]:
import os
from google.colab import userdata

openai_api_key = userdata.get('OPENAI_API_KEY')

os.environ['OPENAI_API_KEY'] = openai_api_key

from openai import OpenAI

client = OpenAI(
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [ ]:
!pip install -qqqq transformers accelerate torch  tiktoken

In [ ]:
# @title
import tiktoken

text = "Hello from HiddenLayer! This is an example of how tokens work"

# use a GPT-style encoding
enc = tiktoken.get_encoding("cl100k_base")

token_ids = enc.encode(text)
tokens = [enc.decode([tid]) for tid in token_ids]

print("Text:")
print(text)
print()

print(f"Characters: {len(text)}")
print(f"Tokens: {len(token_ids)}")
print()

print("Tokens:")
print(tokens)
print()

print("Token IDs:")
print(token_ids)


Text:
Hello from HiddenLayer! This is an example of how tokens work

Characters: 61
Tokens: 13

Tokens:
['Hello', ' from', ' Hidden', 'Layer', '!', ' This', ' is', ' an', ' example', ' of', ' how', ' tokens', ' work']

Token IDs:
[9906, 505, 35342, 9368, 0, 1115, 374, 459, 3187, 315, 1268, 11460, 990]


In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

text = "life is like a box of"
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
next_token_logits = logits[0, -1]

log_probs = F.log_softmax(next_token_logits, dim=-1)

top_k = 5
values, indices = torch.topk(log_probs, top_k)

print("Top predictions:")
for v, i in zip(values, indices):
    token = tokenizer.decode([i])
    prob = torch.exp(v).item() * 100
    print(f"{token!r}: logprob={v.item():.4f}, prob={prob:.2f}%")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Top predictions:
' chocolate': logprob=-3.8633, prob=2.10%
' ice': logprob=-3.8633, prob=2.10%
' candy': logprob=-4.2383, prob=1.44%
' popcorn': logprob=-4.3633, prob=1.27%
' cereal': logprob=-4.5508, prob=1.06%
